# 03 — Silver Transformations

Reads from bronze VARIANT tables and builds normalized silver tables using:
- `data:field::type` extraction syntax on VARIANT columns
- MD5 surrogate keys for deduplication
- INSERT OVERWRITE for idempotent full refresh (MERGE not supported via Databricks Connect)
- PK/FK constraints (RELY) for query optimization and ERD visualization
- Liquid clustering on deals and campaigns

**Tables created:**
- `athletes` — PK: `athlete_sk = MD5(athlete_id)`
- `sponsors` — PK: `sponsor_sk = MD5(sponsor_id)`
- `deals` — PK: `deal_sk = MD5(deal_id)`
- `campaigns` — PK: `campaign_sk = MD5(campaign_id)`

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "nil_sponsorships")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"

print(f"Bronze: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver: {UC_CATALOG}.{SILVER_SCHEMA}")

Bronze: alexander_booth.nil_sponsorships_bronze
Silver: alexander_booth.nil_sponsorships_silver


In [2]:
def add_pk(table: str, name: str, cols: list):
    """Add a primary key constraint if it doesn't already exist."""
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  PK '{name}' already exists on {table}")
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  Added PK '{name}' on {table}")


def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    """Add a foreign key constraint if it doesn't already exist."""
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  FK '{name}' already exists on {table}")
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")
    print(f"  Added FK '{name}' on {table}")

print("Helpers loaded")

Helpers loaded


## Athletes

In [3]:
silver_athletes = f"{UC_CATALOG}.{SILVER_SCHEMA}.athletes"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_athletes} (
        athlete_sk           STRING NOT NULL,
        athlete_id           STRING,
        first_name           STRING,
        last_name            STRING,
        email                STRING,
        phone                STRING,
        school               STRING,
        conference           STRING,
        sport                STRING,
        position             STRING,
        year                 STRING,
        instagram_followers  INT,
        tiktok_followers     INT,
        twitter_followers    INT,
        total_followers      INT,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    INSERT OVERWRITE {silver_athletes}
    SELECT
        MD5(data:athlete_id::string) AS athlete_sk,
        data:athlete_id::string          AS athlete_id,
        data:first_name::string          AS first_name,
        data:last_name::string           AS last_name,
        data:email::string               AS email,
        data:phone::string               AS phone,
        data:school::string              AS school,
        data:conference::string          AS conference,
        data:sport::string               AS sport,
        data:position::string            AS position,
        data:year::string                AS year,
        data:instagram_followers::int    AS instagram_followers,
        data:tiktok_followers::int       AS tiktok_followers,
        data:twitter_followers::int      AS twitter_followers,
        (data:instagram_followers::int + data:tiktok_followers::int + data:twitter_followers::int) AS total_followers,
        _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.athletes_raw
    WHERE data:athlete_id IS NOT NULL
""")

count = spark.table(silver_athletes).count()
print(f"Athletes: {count:,} rows")

Athletes: 200 rows


## Sponsors

In [4]:
silver_sponsors = f"{UC_CATALOG}.{SILVER_SCHEMA}.sponsors"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_sponsors} (
        sponsor_sk           STRING NOT NULL,
        sponsor_id           STRING,
        brand_name           STRING,
        industry             STRING,
        budget_tier          STRING,
        region               STRING,
        contact_name         STRING,
        contact_email        STRING,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    INSERT OVERWRITE {silver_sponsors}
    SELECT
        MD5(data:sponsor_id::string) AS sponsor_sk,
        data:sponsor_id::string      AS sponsor_id,
        data:brand_name::string      AS brand_name,
        data:industry::string        AS industry,
        data:budget_tier::string     AS budget_tier,
        data:region::string          AS region,
        data:contact_name::string    AS contact_name,
        data:contact_email::string   AS contact_email,
        _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.sponsors_raw
    WHERE data:sponsor_id IS NOT NULL
""")

count = spark.table(silver_sponsors).count()
print(f"Sponsors: {count:,} rows")

Sponsors: 50 rows


## Deals

In [5]:
silver_deals = f"{UC_CATALOG}.{SILVER_SCHEMA}.deals"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_deals} (
        deal_sk              STRING NOT NULL,
        deal_id              STRING,
        athlete_id           STRING,
        sponsor_id           STRING,
        deal_type            STRING,
        deal_amount          DOUBLE,
        status               STRING,
        start_date           DATE,
        end_date             DATE,
        created_at           TIMESTAMP,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (status, start_date)
""")

spark.sql(f"""
    INSERT OVERWRITE {silver_deals}
    SELECT
        MD5(data:deal_id::string)    AS deal_sk,
        data:deal_id::string         AS deal_id,
        data:athlete_id::string      AS athlete_id,
        data:sponsor_id::string      AS sponsor_id,
        data:deal_type::string       AS deal_type,
        data:deal_amount::double     AS deal_amount,
        data:status::string          AS status,
        data:start_date::date        AS start_date,
        data:end_date::date          AS end_date,
        data:created_at::timestamp   AS created_at,
        _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.deals_raw
    WHERE data:deal_id IS NOT NULL
""")

count = spark.table(silver_deals).count()
print(f"Deals: {count:,} rows")

Deals: 300 rows


## Campaigns

In [6]:
silver_campaigns = f"{UC_CATALOG}.{SILVER_SCHEMA}.campaigns"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_campaigns} (
        campaign_sk          STRING NOT NULL,
        campaign_id          STRING,
        deal_id              STRING,
        platform             STRING,
        campaign_date        DATE,
        impressions          INT,
        engagements          INT,
        clicks               INT,
        conversions          INT,
        spend_amount         DOUBLE,
        engagement_rate      DOUBLE,
        click_through_rate   DOUBLE,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (platform, campaign_date)
""")

spark.sql(f"""
    INSERT OVERWRITE {silver_campaigns}
    SELECT
        MD5(data:campaign_id::string)    AS campaign_sk,
        data:campaign_id::string         AS campaign_id,
        data:deal_id::string             AS deal_id,
        data:platform::string            AS platform,
        data:campaign_date::date         AS campaign_date,
        data:impressions::int            AS impressions,
        data:engagements::int            AS engagements,
        data:clicks::int                 AS clicks,
        data:conversions::int            AS conversions,
        data:spend_amount::double        AS spend_amount,
        ROUND(data:engagements::double / NULLIF(data:impressions::double, 0), 4) AS engagement_rate,
        ROUND(data:clicks::double / NULLIF(data:engagements::double, 0), 4) AS click_through_rate,
        _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.campaigns_raw
    WHERE data:campaign_id IS NOT NULL
""")

count = spark.table(silver_campaigns).count()
print(f"Campaigns: {count:,} rows")

Campaigns: 800 rows


## Add PK/FK Constraints

In [7]:
print("Adding primary key constraints...")
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.athletes",  "athletes_pk",  ["athlete_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.sponsors",  "sponsors_pk",  ["sponsor_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.deals",     "deals_pk",     ["deal_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.campaigns", "campaigns_pk", ["campaign_sk"])

print("\nAll constraints applied.")

Adding primary key constraints...
  Added PK 'athletes_pk' on alexander_booth.nil_sponsorships_silver.athletes
  Added PK 'sponsors_pk' on alexander_booth.nil_sponsorships_silver.sponsors
  Added PK 'deals_pk' on alexander_booth.nil_sponsorships_silver.deals
  Added PK 'campaigns_pk' on alexander_booth.nil_sponsorships_silver.campaigns

All constraints applied.


In [8]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

for table in ["athletes", "sponsors", "deals", "campaigns"]:
    full = f"{UC_CATALOG}.{SILVER_SCHEMA}.{table}"
    count = spark.table(full).count()
    print(f"  {full}: {count:,} rows")

print("\nDuplicate SK check:")
for table, pk in [("athletes", "athlete_sk"), ("sponsors", "sponsor_sk"),
                  ("deals", "deal_sk"), ("campaigns", "campaign_sk")]:
    full = f"{UC_CATALOG}.{SILVER_SCHEMA}.{table}"
    dupes = spark.sql(f"SELECT {pk}, COUNT(*) c FROM {full} GROUP BY {pk} HAVING c > 1").count()
    status = "PASS" if dupes == 0 else f"FAIL ({dupes} duplicates)"
    print(f"  {table}.{pk}: {status}")

print("\nSilver layer complete.")

SILVER LAYER SUMMARY
  alexander_booth.nil_sponsorships_silver.athletes: 200 rows
  alexander_booth.nil_sponsorships_silver.sponsors: 50 rows
  alexander_booth.nil_sponsorships_silver.deals: 300 rows
  alexander_booth.nil_sponsorships_silver.campaigns: 800 rows

Duplicate SK check:
  athletes.athlete_sk: PASS
  sponsors.sponsor_sk: PASS
  deals.deal_sk: PASS
  campaigns.campaign_sk: PASS

Silver layer complete.
